In [1]:
import torch
from torch import float64
from torch.functional import F
import torchvision

In [15]:
class NeuralNetwork():
    def __init__(self, input_features, hidden_layer1, hidden_layer2, num_classes, bias_value, learning_rate) -> None:
        self.num_classes = num_classes
        self.learning_rate = learning_rate

        self.W1 = torch.randn((input_features, hidden_layer1), dtype=torch.float32) * (2 / input_features) ** 0.5
        self.W2 = torch.randn((hidden_layer1, hidden_layer2), dtype=torch.float32) * (2 / hidden_layer1) ** 0.5
        self.W3 = torch.randn((hidden_layer2, num_classes), dtype=torch.float32) * (2 / hidden_layer2) ** 0.5

        self.b1 = torch.full((hidden_layer1,), bias_value, dtype=torch.float32)
        self.b2 = torch.full((hidden_layer2,), bias_value, dtype=torch.float32)
        self.b3 = torch.full((num_classes,), bias_value, dtype=torch.float32)
    
    def tanh(self, t):
        return torch.tanh(t)
    
    def tanh_grad(self, t):
        return 1 - t**2

    def relu(self, r):
        return torch.relu(r)
    
    def relu_grad(self, r):
        return torch.where(r > 0, 1.0, 0.0)
    
    def softmax(self, s):
        return torch.softmax(s, dim = 1)
    
    def accuracy(self, pred, y):
        labels = torch.argmax(pred, dim = 1)
        return torch.mean((labels == y).float())

    def loss(self, pred, y):
        N = y.shape[0]
        one_hot_y = F.one_hot(y, self.num_classes).float()

        return -torch.sum(one_hot_y * torch.log(pred)) / N

    def forward(self, X):
        
        H1 = X @ self.W1 + self.b1 # (N, input_features) @ (input_feature, hidden_layer1) = (N, hidden_layer1)
        A1 = self.tanh(H1) # (N, hidden_layer1)

        H2 = A1 @ self.W2 + self.b2 # (N, hidden_layer1) @ (hidden_layer1, hidden_layer2) = (N, hidden_layer2)
        A2 = self.relu(H2) # (N, hidden_layer2)

        H3 = A2 @ self.W3 + self.b3 # (N, hidden_layer2) @ (hidden_layer2, num_classes) = (N, num_classes)

        predictions = self.softmax(H3) # (N, num_classes)

        cache = (H1, A1, H2, A2, H3)
        return predictions, cache
    
    def simple_parameter_update(self, parameter, grad, learning_rate):
        new_parameter = parameter - learning_rate * grad
        return new_parameter
    
    def update_parameters(self, parameter, grad, learning_rate, config = None, strategy = 'simple'):
        if strategy == 'simple':
            return self.simple_parameter_update(parameter, grad, learning_rate)
    
    def backward(self, X, y, pred, cache):
        N = y.shape[0]
        H1, A1, H2, A2, H3 = cache

        one_hot_y = F.one_hot(y, num_classes = self.num_classes).float()

        dH3 = (pred - one_hot_y) / N # (N, num_classes)

        dW3 = A2.T @ dH3 # (hidden_layer2, N) @ (N, num_classes) = (hidden_layer2, num_classes)
        db3 = torch.sum(dH3, dim = 0) # (num_classes)
        dA2 = dH3 @ self.W3.T # (N, num_classes) @ (num_classes, hidden_layer2) = (N, hidden_layer2)

        dH2 = self.relu_grad(H2) * dA2 # (N, hidden_layer2)

        dW2 = A1.T @ dH2 # (hidden_layer1, N) @ (N, hidden_layer2) = (hidden_layer1, hidden_layer2)
        db2 = torch.sum(dH2, dim = 0) # (hidden_layer2)
        dA1 = dH2 @ self.W2.T # (N, hidden_layer2) @ (hidden_layer2, hidden_layer1) = (N, hidden_layer1)

        dH1 = self.tanh_grad(A1) * dA1 # (N, hidden_layer1)

        dW1 = X.T @ dH1 # (input_features, N) @ (N, hidden_layer1) = (input_features, hidden_layer1)
        db1 = torch.sum(dH1, dim = 0) # (hidden_layer1)
        
        self.W1 = self.update_parameters(self.W1, dW1, self.learning_rate)
        self.W2 = self.update_parameters(self.W2, dW2, self.learning_rate)
        self.W3 = self.update_parameters(self.W3, dW3, self.learning_rate)

        self.b1 = self.update_parameters(self.b1, db1, self.learning_rate)
        self.b2 = self.update_parameters(self.b2, db2, self.learning_rate)
        self.b3 = self.update_parameters(self.b3, db3, self.learning_rate)


    def train(self, X_train, y_train, X_val, y_val, iterations = 500):
        for iter in range(iterations):
            train_predictions, train_cache = self.forward(X_train)
            training_loss = self.loss(train_predictions, y_train)
            training_accuracy = self.accuracy(train_predictions, y_train)

            val_predictions, val_cache = self.forward(X_val)
            validation_loss = self.loss(val_predictions, y_val)
            validation_accuracy = self.accuracy(val_predictions, y_val)

            print(f"Train loss: {training_loss.item():.2f}, val loss: {validation_loss.item():.2f}")
            print(f"Train accuracy: {training_accuracy.item():.2f}, val acc: {validation_accuracy.item():.2f}")
            self.backward(X_train, y_train, train_predictions, train_cache)

In [3]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATA_DIR = "./data"
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])

train_ds = datasets.MNIST(root=DATA_DIR, train=True, download=False, transform=mnist_transform)
test_ds = datasets.MNIST(root=DATA_DIR, train=False, download=False, transform=mnist_transform)
train_loader = DataLoader(train_ds, batch_size=len(train_ds), shuffle=True)
test_loader = DataLoader(test_ds, batch_size=len(test_ds), shuffle=False)

In [4]:
from sklearn.model_selection import train_test_split

X_train_val, y_train_val = next(iter(train_loader))
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2)

X_test, y_test = next(iter(test_loader))
print(f"train shape: {X_train.shape}, val shape: {X_val.shape}, test shape: {X_test.shape}")

train shape: torch.Size([48000, 784]), val shape: torch.Size([12000, 784]), test shape: torch.Size([10000, 784])


In [20]:
NN = NeuralNetwork(X_train.shape[1], 256, 256, 10, 0.0, 1e-2)
NN.train(X_train, y_train, X_val, y_val, iterations = 200)

Train loss: 2.43, val loss: 2.44
Train accuracy: 0.12, val acc: 0.11
Train loss: 2.40, val loss: 2.41
Train accuracy: 0.12, val acc: 0.12
Train loss: 2.37, val loss: 2.38
Train accuracy: 0.13, val acc: 0.13
Train loss: 2.35, val loss: 2.35
Train accuracy: 0.13, val acc: 0.13
Train loss: 2.32, val loss: 2.33
Train accuracy: 0.14, val acc: 0.14
Train loss: 2.30, val loss: 2.30
Train accuracy: 0.15, val acc: 0.15
Train loss: 2.28, val loss: 2.28
Train accuracy: 0.16, val acc: 0.16
Train loss: 2.26, val loss: 2.26
Train accuracy: 0.17, val acc: 0.17
Train loss: 2.23, val loss: 2.24
Train accuracy: 0.18, val acc: 0.18
Train loss: 2.21, val loss: 2.22
Train accuracy: 0.19, val acc: 0.19
Train loss: 2.19, val loss: 2.20
Train accuracy: 0.20, val acc: 0.20
Train loss: 2.17, val loss: 2.18
Train accuracy: 0.21, val acc: 0.21
Train loss: 2.16, val loss: 2.16
Train accuracy: 0.22, val acc: 0.22
Train loss: 2.14, val loss: 2.14
Train accuracy: 0.24, val acc: 0.24
Train loss: 2.12, val loss: 2.12
T

In [21]:
test_pred, test_cache = NN.forward(X_test)
NN.accuracy(test_pred, y_test)

tensor(0.8315)